# EEG_27 — Few-Shot Calibration: chiudere il gap S-Indep / S-Spec

**Motivazione** (Ko et al. arXiv 2025): con ≤10% dei trial del soggetto target,  
il fine-tuning di un modello S-Indep pre-addestrato chiude significativamente il gap verso S-Spec.

**Protocollo**:
1. **Pre-training** S-Indep: DHSLP addestrato su SUBJ_TRAIN (0–49) — uguale a EEG_13
2. **Calibration**: per ogni soggetto di test (60–73), prendi K trial casuali (support set)
   → fine-tune encoder+classifier per N epoche brevi → valuta sui trial rimanenti
3. **Ablation K**: K ∈ {5, 10, 20, 55} trial  
   (~0.5%, ~1%, ~2%, ~5% dei trial totali per soggetto — 110 parole × 5 sessioni = 550 trial)

**Confronto a 3 livelli**:
```
S-Indep (EEG_13) ← lower bound
Few-shot K=5,10,20,55  ← questa cella
S-Spec (EEG_13b) ← upper bound
```

**Analisi per fenotipo**: C0 vs C1 — chi beneficia di più dalla calibration?  
Ipotesi: C1 (tratto stabile) beneficia di più, C0 (rumore) meno.

**Target EEG_27**: rispondere a 'bastano 10 trial per avvicinare un nuovo soggetto al suo upper bound?'

In [1]:
import json, logging, re, copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from sklearn.metrics import balanced_accuracy_score
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg27')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

# ---- CONFIG ----
N_CHANNELS   = 61
N_SAMPLES    = 384
N_CLASSES    = 4
CLUSTER_SCHEME = 'concr4'

# Split standard
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# DHSLP params (identici a EEG_13 — best config E16_d64)
K_WINDOWS    = 8
N_EDGES      = 16
D_MODEL      = 64
HIDDEN       = 128
N_LAYERS     = 2
DROPOUT      = 0.3
T_WIN        = N_SAMPLES // K_WINDOWS

# Pre-training
LR_PRETRAIN    = 1e-3
BATCH_SIZE     = 64
MAX_EPOCHS_PT  = 60
PATIENCE_PT    = 12
LABEL_SMOOTHING = 0.1
USE_INSTANCE_NORM = True
DATA_METRIC    = 'abs_pcc'

# Few-shot calibration
K_SHOTS      = [5, 10, 20, 55]   # trial per soggetto target
N_SEEDS      = 5                  # ripetizioni random (diversi support set)
LR_FINETUNE  = 1e-4               # LR bassa per non distruggere il pre-training
EPOCHS_FT    = 20                 # epoche fine-tuning (brevi)
BATCH_FT     = 16                 # batch piccolo per K piccoli

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k,v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}

log.info(f'K_SHOTS={K_SHOTS}  N_SEEDS={N_SEEDS}  LR_FT={LR_FINETUNE}  EPOCHS_FT={EPOCHS_FT}')

/home/daniele_u/miniconda3/envs/daniele_311/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
23:02:17 INFO     K_SHOTS=[5, 10, 20, 55]  N_SEEDS=5  LR_FT=0.0001  EPOCHS_FT=20


## §2 — Cluster labels C0/C1

In [2]:
cluster_df = pd.read_csv(project_root / 'figures' / 'eeg08c_subject_clusters.csv')
sid2cluster = dict(zip(cluster_df.subj_id, cluster_df.cluster_k2))  # 0=C0, 1=C1
C0_TEST = sorted([s for s in SUBJ_TEST if sid2cluster.get(s) == 0])
C1_TEST = sorted([s for s in SUBJ_TEST if sid2cluster.get(s) == 1])
log.info(f'Test: C0={C0_TEST} (n={len(C0_TEST)})  C1={C1_TEST} (n={len(C1_TEST)})')

23:02:17 INFO     Test: C0=[60, 63, 65, 66, 68, 70, 72] (n=7)  C1=[61, 62, 64, 67, 69, 71, 73] (n=7)


## §3 — Dataset

Due dataset:
- `EEGRawDataset`: per il pre-training S-Indep (identico a EEG_13)
- `EEGSubjectDataset`: per ogni soggetto di test — restituisce tutti i trial con indice per fare split support/query

In [3]:
class EEGRawDataset(Dataset):
    """Dataset S-Indep per pre-training (identico a EEG_13)."""
    def __init__(self, subj_ids, metric=DATA_METRIC):
        root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
        self.paths, self.labels = [], []
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m or int(m.group(1)) not in subj_ids: continue
            d = torch.load(p, weights_only=False)
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(y_word)
            if c is None: continue
            self.paths.append(p); self.labels.append(c)
        log.info(f'  {len(self.paths)} trial | {len(subj_ids)} soggetti')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        d = torch.load(self.paths[idx], weights_only=False)
        x = d['x'].float()
        if USE_INSTANCE_NORM:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        return x, torch.tensor(self.labels[idx], dtype=torch.long)


class EEGSubjectDataset(Dataset):
    """Dataset per UN soggetto — tutti i trial disponibili per calibration + query."""
    def __init__(self, subj_id, metric=DATA_METRIC):
        root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
        self.paths, self.labels = [], []
        self.subj_id = subj_id
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m or int(m.group(1)) != subj_id: continue
            d = torch.load(p, weights_only=False)
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(y_word)
            if c is None: continue
            self.paths.append(p); self.labels.append(c)

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        d = torch.load(self.paths[idx], weights_only=False)
        x = d['x'].float()
        if USE_INSTANCE_NORM:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        return x, torch.tensor(self.labels[idx], dtype=torch.long)


def make_pretrain_loaders():
    tr = EEGRawDataset(SUBJ_TRAIN)
    va = EEGRawDataset(SUBJ_VAL)
    labels   = np.array(tr.labels)
    counts   = np.bincount(labels, minlength=N_CLASSES)
    sample_w = torch.tensor(1.0 / counts[labels], dtype=torch.float)
    sampler  = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(tr, BATCH_SIZE, sampler=sampler, **kw),
            DataLoader(va, BATCH_SIZE, shuffle=False, **kw))

## §4 — Modello DHSLP (identico a EEG_13, best config E16_d64)

In [4]:
class HGNNConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias   = nn.Parameter(torch.zeros(out_ch))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, X, H):
        d_v = H.sum(dim=2).clamp(min=1e-6)
        d_e = H.sum(dim=1).clamp(min=1e-6)
        Dv  = (1.0 / d_v.sqrt()).unsqueeze(-1)
        De  = (1.0 / d_e).unsqueeze(1)
        out = Dv * (X @ self.weight)
        out = torch.bmm(H.transpose(1, 2), out)
        out = De.transpose(1, 2) * out
        out = torch.bmm(H, out)
        return Dv * out + self.bias


class DHSLP(nn.Module):
    def __init__(self, n_nodes=N_CHANNELS, T_win=T_WIN, K=K_WINDOWS,
                 n_edges=N_EDGES, d_model=D_MODEL, hidden=HIDDEN,
                 n_classes=N_CLASSES, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.K, self.T_win, self.d_model = K, T_win, d_model
        self.E       = nn.Parameter(torch.randn(n_edges, d_model) * 0.01)
        self.pos_enc = nn.Parameter(torch.randn(n_nodes, d_model) * 0.01)
        self.node_proj = nn.Sequential(
            nn.Linear(T_win, d_model), nn.LayerNorm(d_model), nn.ELU()
        )
        dims = [d_model] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop  = nn.Dropout(dropout)
        self.clf   = nn.Linear(hidden, n_classes)

    def forward(self, x):
        B, N, _ = x.shape
        outs = []
        for k in range(self.K):
            x_k  = x[:, :, k*self.T_win:(k+1)*self.T_win]
            feat = self.node_proj(x_k) + self.pos_enc
            scores = torch.matmul(feat, self.E.T) / (self.d_model ** 0.5)
            H_k  = torch.softmax(scores, dim=2)
            out  = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k)
                out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out)
                out = self.drop(out)
            outs.append(out.mean(dim=1))
        return self.clf(torch.stack(outs, dim=1).mean(dim=1))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info(f'device: {device}')
_m = DHSLP()
assert _m(torch.randn(4, N_CHANNELS, N_SAMPLES)).shape == (4, N_CLASSES)
n_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
log.info(f'DHSLP OK — {n_p:,} param'); del _m

23:02:17 INFO     device: cuda
23:02:17 INFO     DHSLP OK — 34,052 param


## §5 — Pre-training S-Indep

Addestra DHSLP su SUBJ_TRAIN (0–49).  
Salva il checkpoint per il fine-tuning in §6.  
**Identico a EEG_13 E16_d64** — se hai già il checkpoint, caricalo direttamente.

In [ ]:
CHECKPOINT_PATH = project_root / 'data' / 'eeg27_dhslp_pretrained.pt'

def run_epoch(model, loader, optimizer=None, criterion=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, all_lbl, all_pred = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss   = criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            all_lbl.extend(y.cpu().numpy())
            all_pred.extend(logits.argmax(1).cpu().numpy())
    bacc = balanced_accuracy_score(all_lbl, all_pred)
    return total_loss / len(loader.dataset), bacc, np.array(all_lbl), np.array(all_pred)


if CHECKPOINT_PATH.exists():
    log.info(f'Carico checkpoint pre-addestrato da {CHECKPOINT_PATH}')
    pretrained_state = torch.load(CHECKPOINT_PATH, map_location='cpu')
    _model_check = DHSLP().to(device)
    _model_check.load_state_dict(pretrained_state)
    log.info('Checkpoint caricato OK')
    del _model_check
else:
    log.info('Nessun checkpoint trovato — avvio pre-training S-Indep')
    tr_l, va_l = make_pretrain_loaders()
    model_pt = DHSLP().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    opt   = torch.optim.Adam(model_pt.parameters(), lr=LR_PRETRAIN, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS_PT)

    run_pt = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                        name=f'eeg27_pretrain_sindep_{CLUSTER_SCHEME}',
                        config=dict(notebook='EEG_27', phase='pretrain',
                                    n_edges=N_EDGES, d_model=D_MODEL,
                                    lr=LR_PRETRAIN, batch_size=BATCH_SIZE),
                        reinit='finish_previous',
                        settings=wandb.Settings(start_method='thread'))

    best_val, best_state, patience_cnt = 0.0, None, 0
    for epoch in range(1, MAX_EPOCHS_PT + 1):
        tr_l2, va_l2 = run_epoch(model_pt, tr_l, opt, criterion), run_epoch(model_pt, va_l, criterion=criterion)
        tr_loss, tr_b = tr_l2[0], tr_l2[1]
        va_loss, va_b = va_l2[0], va_l2[1]
        sched.step()
        run_pt.log({'train/loss': tr_loss, 'train/bacc': tr_b, 'val/loss': va_loss, 'val/bacc': va_b, 'epoch': epoch})
        if va_b > best_val:
            best_val = va_b
            best_state = {k: v.cpu().clone() for k, v in model_pt.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE_PT:
            log.info(f'Early stop epoch {epoch}'); break

    run_pt.summary['val_bacc'] = best_val
    run_pt.finish()
    pretrained_state = best_state
    torch.save(pretrained_state, CHECKPOINT_PATH)
    log.info(f'Pre-training done. val_bacc={best_val:.4f}. Checkpoint salvato in {CHECKPOINT_PATH}')

23:02:17 INFO     Nessun checkpoint trovato — avvio pre-training S-Indep


### §5b — Baseline S-Indep (zero-shot)

Valuta il modello pre-addestrato sui soggetti di test **senza calibration**.  
Questa è la baseline S-Indep che il few-shot deve battere.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

sindep_per_subj = {}   # sid → bacc
for sid in SUBJ_TEST:
    ds = EEGSubjectDataset(sid)
    if len(ds) == 0:
        log.warning(f'P{sid:03d}: nessun trial trovato, skip'); continue
    loader = DataLoader(ds, BATCH_SIZE, shuffle=False, num_workers=2)
    model_eval = DHSLP().to(device)
    model_eval.load_state_dict(pretrained_state)
    _, bacc, _, _ = run_epoch(model_eval, loader, criterion=criterion)
    sindep_per_subj[sid] = bacc
    log.info(f'  P{sid:03d} S-Indep bAcc={bacc:.4f} (cluster={"C0" if sid2cluster.get(sid)==0 else "C1"})')

SINDEP_MEAN = np.mean(list(sindep_per_subj.values()))
print(f'\nS-Indep medio: {SINDEP_MEAN:.4f}')

## §6 — Few-Shot Calibration

Per ogni soggetto di test e per ogni K ∈ K_SHOTS:
1. Estrai K trial casuali (support set) → fine-tune
2. Valuta sui trial rimanenti (query set)
3. Ripeti N_SEEDS volte → media e std

**Strategia fine-tuning** (Ko et al. 2025):
- LR bassa (1e-4) per non distruggere il pre-training
- Poche epoche (20) — con K piccolo si overfitta subito
- Opzionale: freeze encoder, fine-tune solo la testa (`model.clf`)

In [ ]:
def fewshot_calibrate(sid, k_shot, seed=42, freeze_encoder=False):
    """
    Calibra il modello S-Indep con k_shot trial del soggetto sid.
    Restituisce bAcc sul query set (trial non usati per il fine-tuning).
    """
    ds = EEGSubjectDataset(sid)
    if len(ds) < k_shot + 10:
        log.warning(f'P{sid:03d}: troppo pochi trial ({len(ds)}) per k={k_shot}')
        return np.nan

    rng = np.random.default_rng(seed)
    all_idx = np.arange(len(ds))
    support_idx = rng.choice(all_idx, k_shot, replace=False)
    query_idx   = np.setdiff1d(all_idx, support_idx)

    support_ds = Subset(ds, support_idx)
    query_ds   = Subset(ds, query_idx)

    # DataLoader con batch piccolo (adattato a k_shot)
    batch_sup = min(BATCH_FT, k_shot)
    sup_loader = DataLoader(support_ds, batch_sup, shuffle=True,  num_workers=0)
    qry_loader = DataLoader(query_ds,  BATCH_SIZE, shuffle=False, num_workers=2)

    # Copia del modello pre-addestrato (non modifica lo stato globale)
    model_ft = DHSLP().to(device)
    model_ft.load_state_dict(pretrained_state)

    if freeze_encoder:
        # Freeze tutto tranne la testa finale
        for name, param in model_ft.named_parameters():
            if 'clf' not in name:
                param.requires_grad = False

    opt_ft = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_ft.parameters()),
        lr=LR_FINETUNE, weight_decay=1e-5
    )
    crit_ft = nn.CrossEntropyLoss()

    # Fine-tuning
    model_ft.train()
    for _ in range(EPOCHS_FT):
        for x_sup, y_sup in sup_loader:
            x_sup, y_sup = x_sup.to(device), y_sup.to(device)
            loss = crit_ft(model_ft(x_sup), y_sup)
            opt_ft.zero_grad(); loss.backward(); opt_ft.step()

    # Valutazione sul query set
    model_ft.eval()
    all_lbl, all_pred = [], []
    with torch.no_grad():
        for x_q, y_q in qry_loader:
            logits = model_ft(x_q.to(device))
            all_lbl.extend(y_q.numpy())
            all_pred.extend(logits.argmax(1).cpu().numpy())
    return balanced_accuracy_score(all_lbl, all_pred)


log.info('Avvio few-shot calibration...')
log.info(f'K_SHOTS={K_SHOTS}  N_SEEDS={N_SEEDS}  Soggetti test={SUBJ_TEST}')

# Struttura risultati: results[k_shot][sid] = lista di bacc per ogni seed
results_fs = {k: {} for k in K_SHOTS}

for k_shot in K_SHOTS:
    log.info(f'\n--- K={k_shot} ---')
    for sid in SUBJ_TEST:
        baccs = []
        for seed in range(N_SEEDS):
            b = fewshot_calibrate(sid, k_shot, seed=seed)
            baccs.append(b)
        results_fs[k_shot][sid] = np.array(baccs)
        cluster_name = 'C0' if sid2cluster.get(sid) == 0 else 'C1'
        log.info(f'  P{sid:03d} [{cluster_name}] K={k_shot}: {np.nanmean(baccs):.4f} ± {np.nanstd(baccs):.4f}')

log.info('\nFew-shot calibration completata.')

## §7 — Confronto: S-Indep / Few-shot K / S-Spec

In [ ]:
# S-Spec upper bound da EEG_13b (aggiornare con valore reale)
# Mappa sid → bacc S-Spec (da eeg13b_subject_ranking.csv se disponibile)
try:
    ranking = pd.read_csv(project_root / 'figures' / 'eeg13b_subject_ranking.csv')
    sspec_map = dict(zip(
        ranking['Subject'].str[1:].astype(int),
        ranking['Test bAcc']
    ))
    log.info(f'S-Spec caricato: {len(sspec_map)} soggetti')
except FileNotFoundError:
    sspec_map = {}
    log.warning('eeg13b_subject_ranking.csv non trovato — S-Spec non disponibile')

# ---- Plot 1: curva K-shot medio ----
chance = 1 / N_CLASSES
k_means = [np.nanmean([results_fs[k][sid].mean() for sid in SUBJ_TEST]) for k in K_SHOTS]
k_stds  = [np.nanstd([results_fs[k][sid].mean() for sid in SUBJ_TEST]) for k in K_SHOTS]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('EEG_27 — Few-Shot Calibration', fontsize=13, fontweight='bold')

ax = axes[0]
ax.errorbar(K_SHOTS, k_means, yerr=k_stds, marker='o', capsize=4, color='steelblue', label='Few-shot')
ax.axhline(SINDEP_MEAN, color='gray',   ls='--', label=f'S-Indep ({SINDEP_MEAN:.3f})')
if sspec_map:
    SSPEC_MEAN = np.mean([sspec_map[sid] for sid in SUBJ_TEST if sid in sspec_map])
    ax.axhline(SSPEC_MEAN,  color='green',  ls='--', label=f'S-Spec ({SSPEC_MEAN:.3f})')
ax.axhline(chance, color='red', ls=':', label=f'Chance ({chance:.0%})')
ax.set_xlabel('K (trial support set)'); ax.set_ylabel('Test bAcc')
ax.set_title('bAcc vs K (media su 14 soggetti test)')
ax.legend(fontsize=8); ax.set_xticks(K_SHOTS)

# ---- Plot 2: C0 vs C1 per K=20 ----
ax2 = axes[1]
K_PLOT = min(20, max(K_SHOTS))  # usa K=20 o il più grande disponibile
if K_PLOT in results_fs:
    c0_baccs = [results_fs[K_PLOT][sid].mean() for sid in C0_TEST if sid in results_fs[K_PLOT]]
    c1_baccs = [results_fs[K_PLOT][sid].mean() for sid in C1_TEST if sid in results_fs[K_PLOT]]
    ax2.boxplot([c0_baccs, c1_baccs], labels=['C0', 'C1'])
    ax2.axhline(SINDEP_MEAN, color='gray', ls='--', label='S-Indep baseline')
    ax2.axhline(chance, color='red', ls=':', label='Chance')
    ax2.set_ylabel('Test bAcc'); ax2.set_title(f'C0 vs C1 — K={K_PLOT}')
    ax2.legend(fontsize=8)
    print(f'C0 K={K_PLOT}: {np.mean(c0_baccs):.4f} ± {np.std(c0_baccs):.4f}')
    print(f'C1 K={K_PLOT}: {np.mean(c1_baccs):.4f} ± {np.std(c1_baccs):.4f}')

# ---- Plot 3: per-soggetto K=55 ----
ax3 = axes[2]
K_FULL = max(K_SHOTS)
sids = sorted(SUBJ_TEST)
fs_vals  = [results_fs[K_FULL].get(sid, np.array([np.nan])).mean() for sid in sids]
si_vals  = [sindep_per_subj.get(sid, np.nan) for sid in sids]
ss_vals  = [sspec_map.get(sid, np.nan) for sid in sids]
colors_pt = ['#EF9A9A' if sid2cluster.get(sid)==0 else '#A5D6A7' for sid in sids]
x = np.arange(len(sids))
ax3.bar(x - 0.25, si_vals,  0.25, label='S-Indep', color='lightblue', alpha=0.8)
ax3.bar(x,        fs_vals,  0.25, label=f'FS K={K_FULL}', color=colors_pt, alpha=0.9)
if any(not np.isnan(v) for v in ss_vals):
    ax3.bar(x + 0.25, ss_vals, 0.25, label='S-Spec',  color='lightgreen', alpha=0.8)
ax3.axhline(chance, color='red', ls=':', label='Chance')
ax3.set_xticks(x); ax3.set_xticklabels([f'P{s}' for s in sids], rotation=45, fontsize=7)
ax3.set_ylabel('Test bAcc'); ax3.set_title(f'Per soggetto — K={K_FULL} (rosso=C0, verde=C1)')
ax3.legend(fontsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg27_fewshot_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Tabella riepilogo ---')
print(f'  S-Indep:   {SINDEP_MEAN:.4f}')
for k, m, s in zip(K_SHOTS, k_means, k_stds):
    delta = m - SINDEP_MEAN
    print(f'  FS K={k:3d}:  {m:.4f} ± {s:.4f}  (Δ vs S-Indep: {delta:+.4f})')
if sspec_map:
    print(f'  S-Spec:    {SSPEC_MEAN:.4f}')

## §8 — Ablation: freeze encoder vs full fine-tuning (opzionale)

Con K piccolo (K=5), fine-tunare solo la testa può essere migliore del full fine-tuning  
(meno parametri → meno overfitting sul support set).

In [ ]:
# Decommentare per eseguire
# K_ABL = 5
# results_freeze = {}
# results_full   = {}
# for sid in SUBJ_TEST:
#     baccs_freeze = [fewshot_calibrate(sid, K_ABL, seed=s, freeze_encoder=True)  for s in range(N_SEEDS)]
#     baccs_full   = [fewshot_calibrate(sid, K_ABL, seed=s, freeze_encoder=False) for s in range(N_SEEDS)]
#     results_freeze[sid] = np.nanmean(baccs_freeze)
#     results_full[sid]   = np.nanmean(baccs_full)
# print(f'Freeze: {np.mean(list(results_freeze.values())):.4f}')
# print(f'Full:   {np.mean(list(results_full.values())):.4f}')

print('Ablation freeze/full non eseguita — decommentare il blocco sopra.')

## §9 — Log W&B risultati aggregati

In [ ]:
run_agg = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=f'eeg27_fewshot_summary_{CLUSTER_SCHEME}',
                     config=dict(
                         notebook='EEG_27', model='DHSLP_fewshot',
                         k_shots=K_SHOTS, n_seeds=N_SEEDS,
                         lr_pretrain=LR_PRETRAIN, lr_finetune=LR_FINETUNE,
                         epochs_ft=EPOCHS_FT, batch_ft=BATCH_FT,
                         n_test_subj=len(SUBJ_TEST),
                     ),
                     reinit='finish_previous',
                     settings=wandb.Settings(start_method='thread'))

run_agg.summary['sindep_mean'] = SINDEP_MEAN
for k, m, s in zip(K_SHOTS, k_means, k_stds):
    run_agg.summary[f'fs_k{k}_mean']  = m
    run_agg.summary[f'fs_k{k}_std']   = s
    run_agg.summary[f'fs_k{k}_delta'] = m - SINDEP_MEAN

# Curva K vs bAcc
for k, m in zip(K_SHOTS, k_means):
    run_agg.log({'k_shot': k, 'fs_bacc_mean': m})

run_agg.log({'fewshot_curve': wandb.Image(str(FIG_DIR / 'eeg27_fewshot_results.png'))})
run_agg.finish()
print('W&B log completato.')

## §10 — Discussione

**Domande di interpretazione**:
1. Quanto salgono le bAcc con K=55? → se Δ > 0.05 rispetto S-Indep = calibration efficace
2. Curva satura a K basso (K=10→20 ~ K=55)? → utile per BCI con poca calibration
3. C1 beneficia di più di C0? → coerente con proficiency come tratto in C1
4. Freeze encoder > full finetune per K piccoli? → regolarizzazione implicita

**Per la tesi** (Cap. 5 — Lavori futuri):
- Se Δ positivo con K piccolo: 'con appena 10 trial il modello si personalizza sul nuovo soggetto —  
  prospettiva realistica per BCI clinici che richiedono calibration minima'
- Se C1 beneficia di più: rinforza la narrativa dei fenotipi (C1 ha un segnale decodificabile,  
  bastano pochi trial per adattarsi a quella firma individuale)
- Citazione chiave: Ko et al. arXiv 2025 (training ciclico + ≤10% target trials)